# Lab 00 — Setup and First Quantum Circuit

Check the Qiskit installation, build a one-qubit circuit, inspect its
statevector, and run it on the Aer simulator.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

## Parameters

In [ ]:
APPLY_X = True   # flip the qubit from |0> to |1>?
SHOTS = 1024
SEED = 7

## Step 1 — Check the environment

In [ ]:
import platform

import matplotlib
import numpy as np
import qiskit
import qiskit_aer

print("Python    :", platform.python_version())
print("Qiskit    :", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)
print("NumPy     :", np.__version__)
print("Matplotlib:", matplotlib.__version__)

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

from common.qlab import format_state, save_circuit

## Step 2 — Build the circuit
Every qubit starts in $|0\rangle$. The $X$ gate (quantum NOT) maps $|0\rangle \to |1\rangle$.

In [ ]:
q = QuantumRegister(1, "q")
c = ClassicalRegister(1, "c")
qc = QuantumCircuit(q, c)

if APPLY_X:
    qc.x(q[0])
qc.barrier(label="State Preparation");

## Step 3 — Statevector before measurement

In [ ]:
state = Statevector(qc.remove_final_measurements(inplace=False))
print("Statevector:", np.round(state.data, 4))
print("Dirac form :", format_state(*state.data))
probabilities = {str(k): round(float(v), 4) for k, v in state.probabilities_dict().items()}
print("Probabilities:", probabilities)

## Step 4 — Measure and simulate

In [ ]:
qc.measure(q[0], c[0])
qc.barrier(label="Measurement")
save_circuit(qc, "circuit.png", scale=1.0);

backend = AerSimulator(seed_simulator=SEED)
counts = backend.run(transpile(qc, backend), shots=SHOTS).result().get_counts()
print("Counts:", counts)

plot_histogram(counts, title="First circuit")

## Verification

In [ ]:
expected = "1" if APPLY_X else "0"
assert counts == {expected: SHOTS}, counts
assert np.isclose(probabilities[expected], 1.0)
print(f"Measured |{expected}> in all {SHOTS} shots. Environment is ready!")

In [ ]:
qc.draw(output="mpl", scale=1.0)